📅 **论文年份 (Year):2016 年**  
*Variational Lossy Autoencoder — Chen et al. (ICLR 2017)*

# Paper 17: Variational Lossy Autoencoder(变分有损自编码器)
## Xi Chen, Diederik P. Kingma, et al. (2016)

### VAE: Generative Model with Learned Latent Space(VAE：具有学习潜在空间的生成模型)

Combines deep learning with variational inference for generative modeling.

将深度学习与变分推断(variational inference)相结合，用于生成式建模。

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的)：** 变分自编码器(VAE)是一种能"学会画画"的模型：它先把图片压缩成一小串数字(潜在编码)，再从这串数字还原出图片。但实际使用中有个尴尬的现象——如果解码器太强大(比如自回归模型 PixelRNN)，它干脆自己把图片全画出来，完全无视潜在编码，编码变成了摆设。这篇论文想弄清楚：为什么会这样？以及如何精确控制潜在编码里到底存什么信息。

**💡 主要贡献：** 论文用信息论(可以理解为"压缩文件的学问")给出了漂亮的解释：只要解码器自己就能预测的信息，模型就不会费力放进潜在编码里。基于这个洞察，作者提出了"变分有损自编码器"(VLAE)——就像 JPEG 有损压缩会丢掉人眼不敏感的细节一样，VLAE 可以故意让解码器只负责局部纹理，从而迫使潜在编码去记住全局结构(轮廓、形状)。此外还引入自回归流(autoregressive flow)让先验分布更灵活，在多个图像数据集上刷新了当时的密度估计纪录。

**🔧 方法：** 核心思路是"分工"：给解码器戴上"近视眼镜"(只能看到局部窗口的自回归解码器)，它擅长画细节纹理却看不清整体；这样全局信息只能存进潜在编码，二者各司其职。同时把原本简单的高斯先验换成用自回归流变换过的灵活先验，让编码空间更贴合数据的真实分布。

**🌟 意义：** 这篇论文让人们第一次清楚理解了 VAE 中"潜在编码被忽略"(posterior collapse)这一著名难题的根源，并示范了如何把表示学习变成可设计、可控制的工程。VAE 由 Kingma 提出后成为与 GAN 并列的两大生成模型流派之一，其"压缩—还原"思想一路影响到今天的 Stable Diffusion(其底层正是在 VAE 的潜在空间中工作)。理解本文，就理解了现代生成式 AI 的一块重要基石。

## 🎯 核心结论 (Key Takeaways)

- **论文核心发现——信息"能省则省"：** 只要强大的自回归解码器自己就能预测出来的局部细节，VAE 就绝不会费力写进潜在编码。这一信息论视角首次讲清了著名的"潜在编码被忽略"(posterior collapse)难题的根源。
- **表示是可以"设计"的：** VLAE 故意给解码器戴上只能看局部小窗口的"近视眼镜"，迫使全局结构(轮廓、形状)只能存入潜在编码；再用自回归流(autoregressive flow)升级先验分布，在 MNIST、CIFAR-10 等数据集上刷新了当时的密度估计纪录。
- **本 notebook 用纯 NumPy 从零实现了一个标准 VAE：** 16 维输入(4x4 图案) → 32 个隐藏单元 → 2 维潜在空间，在 200 张四类图案(横线/竖线/对角线/角块)上完整走通了编码 → 重参数化 (z = μ + σ⊙ε) → 解码的流程，并手写了 ELBO 损失 = 重构 BCE + KL 散度。
- **实验演示了 VAE 的三大标志性能力：** 把 200 张图编码成 2 维散点图观察不同图案的分布；从先验 N(0, I) 直接采样 8 个编码"无中生有"地生成新图；在两张图的编码之间做 8 步线性插值，观察图案的平滑渐变。注意本笔记本用的是未训练的 VAE，重点在演示机制本身，所以重建和生成的图看起来还是模糊噪声——训练后这些能力才会真正显现。
- **重参数化技巧是 VAE 能训练的关键：** 对同一张输入图采样 100 次潜在编码，会得到一团围绕均值 μ、以 2σ 为边界的高斯"云"——把随机性挪到外部噪声 ε 上，μ 和 σ 就变成可求梯度的普通参数。
- **带走信息：** 潜在编码里存什么信息不是玄学，而是可以通过解码器结构精确控制的设计选择——这种"在压缩后的潜空间里做生成"的思想，正是今天 Stable Diffusion 等模型的基石。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为，组件越强，整体就越好——解码器越强大，VAE 学到的表征应该越出色。** 但这篇论文发现恰恰相反：给 VAE 配上强大的自回归解码器(如 PixelRNN)后，解码器干脆自己把图片全画出来，潜变量被彻底"晾在一边"(posterior collapse)——模型局部升级反而让核心的潜在表征彻底失效。论文的信息论解释是：凡是解码器自己能预测的信息，模型就"能省则省"，绝不写进潜在编码。

- **常识认为，往编码过程里"注入随机噪声"是缺陷，会污染信息。** 但重参数化采样 `z = μ + σ·ε` 恰恰是 VAE 的点睛之笔：正是这团刻意加入的噪声，逼着潜在空间中相邻的点解码出相似的图案，才有了本 notebook 中横线到竖线的平滑插值实验。你可以对比看：编码器输出的不是一个点而是一团"云"(重参数化可视化那一节的 100 次采样)，去掉这团云，潜在空间就会退化成支离破碎、无法插值的孤岛。

- **常识认为，"有损压缩"是不得已的妥协，无损才是理想。** 但论文把"有损"变成了可设计的特性：像 JPEG 故意丢弃人眼不敏感的细节一样，VLAE 故意让解码器只看局部小窗口，从而精确控制潜变量"记住全局轮廓、忘掉局部纹理"。丢什么、留什么不再是副作用，而是表示学习的设计旋钮。

- **常识认为，训练目标应该让编码尽可能多地携带输入信息。** 但 VAE 的 KL 散度项(本 notebook 损失函数中与重构损失并列的那一项)却在反向用力——不断把每张图的编码分布往同一个标准正态先验上拉，故意"抹掉"信息。正是这股"往回拉"的力量，让随机采样 z ~ N(0, I) 再解码就能凭空生成新图案(从先验采样那一节)；只顾重构的普通自编码器反而做不到这一点。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库，并固定随机种子，保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算库，负责矩阵运算)和 `matplotlib.pyplot`(画图库)。
- 调用 `np.random.seed(42)` 固定随机数种子——就像掷骰子前先"设定好剧本"，之后所有随机初始化、随机采样每次运行都相同，方便复现实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Variational Autoencoder (VAE) Basics(变分自编码器基础)

VAE learns:
- **Encoder**: q(z|x) - approximate posterior
- **Decoder**: p(x|z) - generative model

**Loss**: ELBO = Reconstruction Loss + KL Divergence

VAE 学习：
- **编码器(Encoder)**：q(z|x) —— 近似后验分布
- **解码器(Decoder)**：p(x|z) —— 生成模型

**损失函数**：ELBO = 重构损失(Reconstruction Loss) + KL 散度(KL Divergence)

#### 💻 代码解读

**做什么:** 用纯 NumPy 从零实现一个完整的变分自编码器(VAE)类，包括编码器、重参数化采样、解码器和损失函数，并创建一个实例。

**怎么做:**
- 先定义两个激活函数：`relu`(负数归零)和 `sigmoid`(把任意数压缩到 0~1 之间，用来输出像素亮度)。
- `VAE.__init__` 初始化两套权重：编码器(`W_enc_h`、`W_mu`、`W_logvar`)把输入压成潜在分布的均值和对数方差；解码器(`W_dec_h`、`W_recon`)把潜在编码还原成图片。好比一个"压缩员"和一个"还原员"。
- `encode` 输出 `mu` 和 `log_var`——不是一个固定编码，而是一个高斯分布的参数(位置和胖瘦)。
- `reparameterize` 实现重参数化技巧：`z = mu + std * epsilon`，把"随机采样"改写成"固定均值 + 噪声缩放"，这样梯度才能顺利回传。
- `loss` 计算两部分损失：重构损失(二元交叉熵，衡量还原得像不像)+ KL 散度(逼着潜在分布向标准正态分布靠拢)。
- 最后创建 `vae = VAE(16, 32, 2)`：输入 16 维(4x4 图片拉平)，隐藏层 32 维，潜在空间只有 2 维(方便后面画图)，并打印网络结构。

In [ ]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    # 先把x裁剪到[-500,500]再取exp,防止exp(大负数取反)数值溢出
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class VAE:
    def __init__(self, input_dim, hidden_dim, latent_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        # Encoder: x -> h -> (mu, log_var)
        # 编码器不直接输出z,而是输出后验分布q(z|x)的参数:均值mu和对数方差log_var
        # 乘0.1做小随机初始化,避免初始激活过大
        self.W_enc_h = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b_enc_h = np.zeros(hidden_dim)
        
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_mu = np.zeros(latent_dim)
        
        # 学习log(sigma^2)而不是sigma本身:取值可正可负,且保证exp后方差恒为正
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder: z -> h -> x_recon
        self.W_dec_h = np.random.randn(latent_dim, hidden_dim) * 0.1
        self.b_dec_h = np.zeros(hidden_dim)
        
        self.W_recon = np.random.randn(hidden_dim, input_dim) * 0.1
        self.b_recon = np.zeros(input_dim)
    
    def encode(self, x):
        """
        Encode input to latent distribution parameters
        
        Returns: mu, log_var of q(z|x)
        """
        # 形状: (batch, input_dim) -> (batch, hidden_dim)
        h = relu(np.dot(x, self.W_enc_h) + self.b_enc_h)
        # 两个并行的线性头,分别输出均值与对数方差,形状均为(batch, latent_dim)
        mu = np.dot(h, self.W_mu) + self.b_mu
        log_var = np.dot(h, self.W_logvar) + self.b_logvar
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """
        Reparameterization trick: z = mu + sigma * epsilon
        where epsilon ~ N(0, I)
        """
        # exp(0.5*log_var)=sqrt(sigma^2)=sigma,把对数方差还原成标准差
        std = np.exp(0.5 * log_var)
        # *mu.shape把shape元组解包成参数;epsilon是与mu同形状的标准正态噪声
        epsilon = np.random.randn(*mu.shape)
        # 核心技巧:随机性全部来自epsilon,z对mu/std可导,梯度才能反传回编码器
        z = mu + std * epsilon
        return z
    
    def decode(self, z):
        """
        Decode latent code to reconstruction
        
        Returns: reconstructed x
        """
        # 形状: (batch, latent_dim) -> (batch, hidden_dim)
        h = relu(np.dot(z, self.W_dec_h) + self.b_dec_h)
        # sigmoid把输出压到(0,1),与二值像素配合二元交叉熵损失
        x_recon = sigmoid(np.dot(h, self.W_recon) + self.b_recon)
        return x_recon
    
    def forward(self, x):
        """
        Full forward pass
        """
        # Encode
        mu, log_var = self.encode(x)
        
        # Sample latent
        z = self.reparameterize(mu, log_var)
        
        # Decode
        x_recon = self.decode(z)
        
        return x_recon, mu, log_var, z
    
    def loss(self, x, x_recon, mu, log_var):
        """
        VAE loss = Reconstruction Loss + KL Divergence
        """
        # Reconstruction loss (binary cross-entropy)
        # 重构项即-log p(x|z);加1e-8防止log(0)得到-inf
        recon_loss = -np.sum(
            x * np.log(x_recon + 1e-8) + 
            (1 - x) * np.log(1 - x_recon + 1e-8)
        )
        
        # KL divergence: KL(q(z|x) || p(z))
        # where p(z) = N(0, I)
        # KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        # 两个高斯之间的KL有闭式解,无需采样;它把后验拉向标准正态先验,起正则作用
        kl_loss = -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))
        
        # 总损失=负ELBO:重构项让编码保留信息,KL项让潜空间规整,二者相互制衡
        return recon_loss + kl_loss, recon_loss, kl_loss

# Create VAE
# 潜空间维度16->2,是有损压缩:模型被迫只保留最本质的变化因子
input_dim = 16  # e.g., 4x4 image flattened
hidden_dim = 32
latent_dim = 2  # 2D for visualization

vae = VAE(input_dim, hidden_dim, latent_dim)
print(f"VAE created:")
print(f"  Input: {input_dim}")
print(f"  Hidden: {hidden_dim}")
print(f"  Latent: {latent_dim}")

## Generate Synthetic Data(生成合成数据)

Simple 4x4 patterns for demonstration

用于演示的简单 4x4 图案

#### 💻 代码解读

**做什么:** 造一批简单的 4x4 黑白小图案作为训练数据，并把前 4 张画出来看看长什么样。

**怎么做:**
- 定义 `generate_patterns` 函数：按序号循环生成 4 种图案——横线、竖线、对角线、左上角方块，就像 4 种不同的"印章"。
- 每张图案再加上一点微小的随机噪声(`np.random.randn * 0.05`)并用 `np.clip` 限制在 0~1，让数据不至于完全一模一样。
- 每张 4x4 图案用 `flatten()` 拉平成 16 维向量，方便喂给 VAE。
- 调用 `generate_patterns(200)` 生成 200 个样本存入 `X_train`，最后用 `plt.subplots` 把前 4 个样本以灰度图形式画出来。

In [ ]:
def generate_patterns(num_samples=100):
    """
    Generate simple 4x4 binary patterns
    """
    data = []
    
    for i in range(num_samples):
        pattern = np.zeros((4, 4))
        
        # 用i%4轮流生成4种基本图案,数据集天然有4个"类别"
        if i % 4 == 0:
            # Horizontal line
            # 切片[1:2,:]选中第1行(保持二维),整行置1
            pattern[1:2, :] = 1
        elif i % 4 == 1:
            # Vertical line
            pattern[:, 2:3] = 1
        elif i % 4 == 2:
            # Diagonal
            np.fill_diagonal(pattern, 1)
        else:
            # Corner square
            pattern[:2, :2] = 1
        
        # Add small noise
        # 加噪声让同类样本略有差异,再clip回[0,1]保持合法像素值
        noise = np.random.randn(4, 4) * 0.05
        pattern = np.clip(pattern + noise, 0, 1)
        
        # 4x4图像拉平成16维向量,匹配VAE的input_dim
        data.append(pattern.flatten())
    
    return np.array(data)

# Generate training data
X_train = generate_patterns(200)

# Visualize samples
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Pattern {i}')
    ax.axis('off')
plt.suptitle('Training Data Samples')
plt.show()

print(f"Generated {len(X_train)} training samples")

## Test Forward Pass and Loss(测试前向传播与损失)

#### 💻 代码解读

**做什么:** 拿一张训练图片走一遍完整的 VAE 前向流程，检查各环节输出和损失值，并对比原图与(未训练的)重构图。

**怎么做:**
- 取第一张图 `x = X_train[0:1]`，调用 `vae.forward(x)` 得到重构图 `x_recon`、潜在分布参数 `mu`、`log_var` 和采样出的编码 `z`。
- 调用 `vae.loss` 算出总损失，并拆开看重构损失(`recon_loss`)和 KL 散度(`kl_loss`)各占多少。
- 打印输入形状、潜在变量取值和三个损失值，确认整条流水线跑得通。
- 左右并排画出原图和重构图——因为模型还没训练，重构图会是一团模糊的灰色，这是正常的"出厂状态"。

In [ ]:
# Test on a single example
# 切片[0:1]而不是[0]:保留batch维,得到形状(1,16)而非(16,)
x = X_train[0:1]
# 一次前向:编码得(mu,log_var) -> 重参数化采样z -> 解码重构
x_recon, mu, log_var, z = vae.forward(x)

# 未训练时权重随机,损失会很大;这里只验证数值流程正确
total_loss, recon_loss, kl_loss = vae.loss(x, x_recon, mu, log_var)

print(f"Forward pass:")
print(f"  Input shape: {x.shape}")
print(f"  Latent mu: {mu}")
print(f"  Latent log_var: {log_var}")
print(f"  Latent z: {z}")
print(f"  Reconstruction shape: {x_recon.shape}")
print(f"\nLosses:")
print(f"  Total: {total_loss:.4f}")
print(f"  Reconstruction: {recon_loss:.4f}")
print(f"  KL Divergence: {kl_loss:.4f}")

# Visualize reconstruction
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(x.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
ax1.set_title('Original')
ax1.axis('off')

ax2.imshow(x_recon.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
ax2.set_title('Reconstruction (Untrained)')
ax2.axis('off')

plt.show()

## Visualize Latent Space(可视化潜在空间)

Since latent_dim=2, we can visualize the learned representation

由于 latent_dim=2，我们可以将学到的表示可视化

#### 💻 代码解读

**做什么:** 把全部 200 张训练图都编码到 2 维潜在空间，画成散点图，看看不同类型的图案在"压缩地图"上落在哪里。

**怎么做:**
- 遍历 `X_train`，对每张图调用 `vae.encode` 取出均值 `mu` 作为该图的潜在坐标，存入 `latent_codes`；同时用 `i % 4` 记录它属于哪种图案类型(`pattern_types`)。
- 用 `plt.scatter` 把所有潜在编码画在二维平面上，颜色按图案类型区分(`cmap='tab10'`)，并加上颜色条。
- 由于潜在空间恰好是 2 维，可以直接可视化；未训练时各类图案会混在一起，训练后同类图案才会聚成一簇一簇——就像整理前后的衣柜。

In [ ]:
# Encode all training data
latent_codes = []
pattern_types = []

for i, x in enumerate(X_train):
    # reshape(1,-1)把(16,)变成(1,16),-1表示该维自动推断
    mu, log_var = vae.encode(x.reshape(1, -1))
    # 只取均值mu作为该样本的潜在表示(不采样,确定性编码)
    latent_codes.append(mu[0])
    # 数据生成时按i%4循环,所以i%4就是图案类别标签
    pattern_types.append(i % 4)

latent_codes = np.array(latent_codes)
pattern_types = np.array(pattern_types)

# Plot latent space
plt.figure(figsize=(10, 8))
# latent_dim=2正好可以直接画散点图,按类别着色观察聚类结构
scatter = plt.scatter(
    latent_codes[:, 0], 
    latent_codes[:, 1], 
    c=pattern_types, 
    cmap='tab10', 
    alpha=0.6,
    s=50
)
plt.colorbar(scatter, label='Pattern Type')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Latent Space (Untrained VAE)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Latent space visualization shows distribution of encoded patterns")

## Sample from Prior and Generate(从先验采样并生成)

Sample z ~ N(0, I) and decode to generate new samples

采样 z ~ N(0, I) 并解码以生成新样本

#### 💻 代码解读

**做什么:** 演示 VAE 的"生成"能力：不看任何真实图片，直接从标准正态先验分布里随机抽潜在编码，再解码成新图片。

**怎么做:**
- 用 `np.random.randn(8, latent_dim)` 从标准正态分布 N(0, I) 中随机抽 8 个 2 维编码 `z_samples`——相当于在潜在地图上随机"扔飞镖"。
- 对每个 `z` 调用 `vae.decode` 把编码还原成 16 维像素向量，收集到 `generated` 列表。
- 用 2 行 4 列的子图把 8 张生成图画出来，每张标题标注对应的 `z` 值。这正是 VAE 作为生成模型的核心玩法：采样→解码→得到新样本。

In [ ]:
# Sample from standard normal prior
# 生成新样本不需要编码器:直接从先验p(z)=N(0,I)采样z
num_samples = 8
z_samples = np.random.randn(num_samples, latent_dim)

# Generate samples
generated = []
for z in z_samples:
    # 只用解码器把z映射回像素空间;KL项训练时把后验拉近先验,才保证这样采样有意义
    x_gen = vae.decode(z.reshape(1, -1))
    generated.append(x_gen[0])

# Visualize generated samples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.imshow(generated[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'z={z_samples[i][:2]}')
    ax.axis('off')

plt.suptitle('Generated Samples from Prior p(z) = N(0, I)', fontsize=14)
plt.tight_layout()
plt.show()

## Interpolation in Latent Space(潜在空间中的插值)

Smoothly interpolate between two points in latent space

在潜在空间中的两个点之间进行平滑插值

#### 💻 代码解读

**做什么:** 在潜在空间里做"渐变动画"：把两张不同类型图案的编码连成一条线，沿途逐点解码，观察一张图如何平滑变形成另一张。

**怎么做:**
- 分别对横线图案 `x1` 和竖线图案 `x2` 调用 `vae.encode`，取出它们的潜在均值 `mu1` 和 `mu2` 作为两个端点。
- 用 `np.linspace(0, 1, 8)` 生成 8 个插值系数 `alpha`，按公式 `z_interp = (1-alpha)*mu1 + alpha*mu2` 线性混合两个编码——好比调色时把两种颜料按不同比例混合。
- 每个中间编码都用 `vae.decode` 解码成图片，横排画出 8 张图并标注 `α` 值。
- 平滑的过渡说明 VAE 的潜在空间是连续的：相邻的点解码出来的图也相似。

In [ ]:
# Encode two different patterns
x1 = X_train[0:1]  # Pattern type 0
x2 = X_train[1:2]  # Pattern type 1

# 只取两个样本的后验均值作为插值端点,忽略方差(下划线表示丢弃)
mu1, _ = vae.encode(x1)
mu2, _ = vae.encode(x2)

# Interpolate
num_steps = 8
interpolated = []

for alpha in np.linspace(0, 1, num_steps):
    # 在潜空间做线性插值:alpha=0时是mu1,alpha=1时是mu2
    z_interp = (1 - alpha) * mu1 + alpha * mu2
    # 解码每个中间点;若潜空间连续,输出会在两种图案间平滑过渡
    x_interp = vae.decode(z_interp)
    interpolated.append(x_interp[0])

# Visualize interpolation
fig, axes = plt.subplots(1, num_steps, figsize=(16, 2))

for i, ax in enumerate(axes):
    ax.imshow(interpolated[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'α={i/(num_steps-1):.2f}')
    ax.axis('off')

plt.suptitle('Latent Space Interpolation', fontsize=14, y=1.1)
plt.tight_layout()
plt.show()

print("Smooth transitions show continuity in latent space")

## Reparameterization Trick Visualization(重参数化技巧可视化)

#### 💻 代码解读

**做什么:** 可视化重参数化技巧的随机性：对同一张输入图采样 100 次潜在编码，展示它们围绕均值 `μ` 形成一团高斯分布的"云"。

**怎么做:**
- 对同一张图 `x` 调用 `vae.encode` 得到固定的 `mu` 和 `log_var`，然后循环 100 次调用 `vae.reparameterize`——每次噪声 `ε` 不同，所以采到的 `z` 都不一样。
- 用散点图画出这 100 个 `z` 样本，再用红色星号标出分布中心 `μ`。
- 根据 `std = exp(0.5 * log_var)` 计算标准差，用三角函数画出一个红色虚线椭圆，表示 2σ 边界(约 95% 的样本会落在里面)。
- 最后打印 `μ`、`σ` 以及 100 个样本的实际均值和标准差，验证采样结果确实符合设定的分布——这就是"编码是一个分布而不是一个点"的直观体现。

In [ ]:
# Show multiple samples from same distribution
x = X_train[0:1]
mu, log_var = vae.encode(x)

# Sample multiple times
# 同一个(mu,log_var)反复采样:每次epsilon不同,z也不同,体现q(z|x)是一个分布
num_samples = 100
z_samples = []
for _ in range(num_samples):
    z = vae.reparameterize(mu, log_var)
    z_samples.append(z[0])

z_samples = np.array(z_samples)

# Plot distribution
plt.figure(figsize=(10, 8))
plt.scatter(z_samples[:, 0], z_samples[:, 1], alpha=0.3, s=20)
plt.scatter(mu[0, 0], mu[0, 1], color='red', s=200, marker='*', label='μ', zorder=5)

# Draw ellipse for 2 standard deviations
std = np.exp(0.5 * log_var[0])
theta = np.linspace(0, 2*np.pi, 100)
# 用参数方程画2倍标准差椭圆:对角高斯约95%的样本落在此范围内
ellipse_x = mu[0, 0] + 2 * std[0] * np.cos(theta)
ellipse_y = mu[0, 1] + 2 * std[1] * np.sin(theta)
plt.plot(ellipse_x, ellipse_y, 'r--', label='2σ boundary', linewidth=2)

plt.xlabel('z₁')
plt.ylabel('z₂')
plt.title('Reparameterization Trick: z = μ + σ ⊙ ε, where ε ~ N(0,I)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"μ = {mu[0]}")
print(f"σ = {std}")
# axis=0沿样本维求统计量,结果应接近mu与sigma,验证采样正确
print(f"Sample mean: {z_samples.mean(axis=0)}")
print(f"Sample std: {z_samples.std(axis=0)}")

## Key Takeaways(要点总结)

### VAE Architecture:(VAE 架构：)
1. **Encoder**: q_φ(z|x) - Maps input to latent distribution
2. **Reparameterization**: z = μ + σ ⊙ ε (enables backprop)
3. **Decoder**: p_θ(x|z) - Generates output from latent code

### Loss Function (ELBO):(损失函数（ELBO）：)
```
L = E[log p(x|z)] - KL(q(z|x) || p(z))
  = Reconstruction Loss - KL Divergence
```

### KL Divergence:(KL 散度：)
- Regularizes latent space to be close to prior p(z) = N(0, I)
- Prevents overfitting
- Ensures smooth latent space

### Reparameterization Trick:(重参数化技巧：)
- Makes sampling differentiable
- z = μ(x) + σ(x) ⊙ ε, where ε ~ N(0, I)
- Gradients flow through μ and σ

### Properties:(性质：)
- **Generative**: Can sample new data
- **Continuous latent space**: Smooth interpolations
- **Probabilistic**: Models uncertainty
- **Disentangled representations**: (with β-VAE, etc.)

### Applications:(应用：)
- Image generation
- Dimensionality reduction
- Semi-supervised learning
- Anomaly detection
- Data augmentation

### Variants:(变体：)
- **β-VAE**: Weighted KL for disentanglement
- **Conditional VAE**: Conditioned generation
- **Hierarchical VAE**: Multiple latent levels
- **VQ-VAE**: Discrete latents

**中文翻译：**

### VAE 架构：
1. **编码器(Encoder)**：q_φ(z|x) —— 将输入映射到潜在分布
2. **重参数化(Reparameterization)**：z = μ + σ ⊙ ε（使反向传播成为可能）
3. **解码器(Decoder)**：p_θ(x|z) —— 从潜在编码生成输出

### 损失函数（ELBO）：
```
L = E[log p(x|z)] - KL(q(z|x) || p(z))
  = 重构损失 - KL 散度
```

### KL 散度：
- 对潜在空间进行正则化，使其接近先验分布 p(z) = N(0, I)
- 防止过拟合
- 保证潜在空间的平滑性

### 重参数化技巧：
- 使采样过程可微分
- z = μ(x) + σ(x) ⊙ ε，其中 ε ~ N(0, I)
- 梯度可以通过 μ 和 σ 传播

### 性质：
- **生成式**：可以采样生成新数据
- **连续的潜在空间**：可实现平滑插值
- **概率化**：对不确定性建模
- **解耦表示(disentangled representations)**：（借助 β-VAE 等）

### 应用：
- 图像生成
- 降维
- 半监督学习
- 异常检测
- 数据增强

### 变体：
- **β-VAE**：加权 KL 以实现解耦
- **条件 VAE(Conditional VAE)**：条件生成
- **层次化 VAE(Hierarchical VAE)**：多层潜在变量
- **VQ-VAE**：离散潜在变量